# AI Stock Agent — Demo Notebook

This notebook demonstrates the AI Stock Agent deployed on AWS Bedrock AgentCore.
It authenticates via Cognito (demonstrating the user auth flow), then invokes the
agent through the AgentCore SDK (`invoke_agent_runtime`) which returns SSE-streamed
responses for each of the 5 required queries.

**Prerequisites:**
- Infrastructure deployed via `terraform apply`
- Cognito test user created (see README)
- AWS credentials configured (for boto3 SDK calls)
- `pip install boto3`

## Cell 1: Configuration

Set the deployment parameters from Terraform outputs.

In [1]:
import os
from pathlib import Path

from dotenv import dotenv_values, load_dotenv

# Load from .env.notebook at the project root (works in Cursor and Jupyter)
env_path = (
    Path(__file__).resolve().parent.parent / ".env.notebook"
    if "__file__" in dir()
    else Path("../.env.notebook")
)
load_dotenv(env_path, override=False)

# Prepend PATH from .env.notebook so credential_process tools (e.g. granted) are found
extra_path = dotenv_values(env_path).get("PATH", "")
if extra_path:
    os.environ["PATH"] = extra_path + os.pathsep + os.environ.get("PATH", "")

# ──────────────────────────────────────────────
# Values loaded from .env.notebook or shell env vars
# ──────────────────────────────────────────────
RUNTIME_ENDPOINT_ARN = os.environ.get("RUNTIME_ENDPOINT_ARN", "")
COGNITO_USER_POOL_ID = os.environ.get("COGNITO_USER_POOL_ID", "")
COGNITO_CLIENT_ID = os.environ.get("COGNITO_CLIENT_ID", "")
AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")

COGNITO_USERNAME = os.environ.get("COGNITO_USERNAME", "testuser@example.com")
COGNITO_PASSWORD = os.environ.get("COGNITO_PASSWORD", "")

assert RUNTIME_ENDPOINT_ARN, "RUNTIME_ENDPOINT_ARN not set — check .env.notebook"
assert COGNITO_USER_POOL_ID, "COGNITO_USER_POOL_ID not set — check .env.notebook"

# Split endpoint ARN into runtime ARN + qualifier (endpoint name)
# e.g. ".../runtime/ID/runtime-endpoint/ENDPOINT_NAME" → runtime ARN + qualifier
parts = RUNTIME_ENDPOINT_ARN.split("/runtime-endpoint/")
AGENT_RUNTIME_ARN = parts[0]  # arn:...:runtime/ID
ENDPOINT_QUALIFIER = parts[1] if len(parts) > 1 else None

print(f"Runtime ARN:  {AGENT_RUNTIME_ARN}")
print(f"Endpoint:     {ENDPOINT_QUALIFIER}")
print(f"User Pool ID: {COGNITO_USER_POOL_ID}")
print(f"Client ID:    {COGNITO_CLIENT_ID}")
print(f"Region:       {AWS_REGION}")

Runtime ARN:  arn:aws:bedrock-agentcore:us-east-1:070017892077:runtime/ai_stock_agent-YOyxf7Cetp
Endpoint:     ai_stock_agent_endpoint
User Pool ID: us-east-1_OOGUW7lA8
Client ID:    r1dtt64k8gn01isoek4vim3cc
Region:       us-east-1


## Cell 2: Helper Functions

- `authenticate()` — Obtain a JWT from Cognito via `InitiateAuth`
- `invoke_agent()` — Call `invoke_agent_runtime` via boto3 SDK with SSE stream parsing

In [2]:
import json
import uuid

import boto3


def authenticate(
    user_pool_id: str = COGNITO_USER_POOL_ID,
    client_id: str = COGNITO_CLIENT_ID,
    username: str = COGNITO_USERNAME,
    password: str = COGNITO_PASSWORD,
    region: str = AWS_REGION,
) -> dict:
    """Authenticate with Cognito and return the full auth result including tokens."""
    client = boto3.client("cognito-idp", region_name=region)
    response = client.initiate_auth(
        ClientId=client_id,
        AuthFlow="USER_PASSWORD_AUTH",
        AuthParameters={
            "USERNAME": username,
            "PASSWORD": password,
        },
    )
    return response["AuthenticationResult"]


def invoke_agent(
    prompt: str,
    id_token: str,
    thread_id: "str | None" = None,
    stream: bool = True,
    runtime_arn: str = AGENT_RUNTIME_ARN,
    qualifier: "str | None" = ENDPOINT_QUALIFIER,
    region: str = AWS_REGION,
) -> str:
    """Invoke the agent via AgentCore SDK and print streaming tokens.

    Uses boto3 invoke_agent_runtime. The response is a StreamingBody
    that our container fills with SSE events (stream=True) or JSON.
    Returns the full concatenated response text.
    """
    thread_id = thread_id or str(uuid.uuid4())
    payload = json.dumps(
        {
            "prompt": prompt,
            "thread_id": thread_id,
            "stream": stream,
        }
    ).encode()

    client = boto3.client("bedrock-agentcore", region_name=region)
    invoke_kwargs = {
        "agentRuntimeArn": runtime_arn,
        "runtimeSessionId": thread_id,
        "contentType": "application/json",
        "accept": "text/event-stream" if stream else "application/json",
        "payload": payload,
    }
    if qualifier:
        invoke_kwargs["qualifier"] = qualifier

    response = client.invoke_agent_runtime(**invoke_kwargs)

    full_response = []
    body = response["response"]  # botocore StreamingBody

    raw = body.read()
    text = raw.decode("utf-8") if isinstance(raw, bytes) else raw

    # Try SSE parsing first (data: {...} lines)
    sse_found = False
    for line in text.splitlines():
        if line.startswith("data: "):
            sse_found = True
            data = json.loads(line[6:])
            if data.get("type") == "token":
                token = data["content"]
                print(token, end="", flush=True)
                full_response.append(token)
            elif data.get("type") == "end":
                break

    if not sse_found:
        # Not SSE — try JSON, then fall back to raw text
        try:
            result = json.loads(text)
            answer = result.get("response", str(result))
        except json.JSONDecodeError:
            answer = text
        print(answer)
        full_response.append(answer)

    print()
    return "".join(full_response)


print("Helpers loaded ✓")

Helpers loaded ✓


## Cell 3: Authenticate with Cognito

Obtain a JWT via `USER_PASSWORD_AUTH` flow.

In [3]:
auth_result = authenticate()
id_token = auth_result["IdToken"]

print(f"Access Token (first 40 chars): {auth_result['AccessToken'][:40]}...")
print(f"ID Token     (first 40 chars): {id_token[:40]}...")
print(f"Token Type: {auth_result['TokenType']}")
print(f"Expires In: {auth_result['ExpiresIn']}s")
print("\nAuthentication successful ✓")

Access Token (first 40 chars): eyJraWQiOiJiSWprdjRRVXlpbkRhamtjdHBOakVE...
ID Token     (first 40 chars): eyJraWQiOiJOK3ZKT3VtemlLK0x2Y3RLOTdSa2pu...
Token Type: Bearer
Expires In: 3600s

Authentication successful ✓


## Cell 4: Query 1 — Real-Time Stock Price

> *"What is the stock price for Amazon right now?"*

Expected: Agent calls `retrieve_realtime_stock_price` with ticker AMZN.

In [4]:
response_1 = invoke_agent(
    "What is the stock price for Amazon right now?",
    id_token=id_token,
)

I'll get

 the current stock price for Amazon (

AMZN) for you.

Amazon

's current

...

026 

at

 18

:58 

UTC.

## Cell 5: Query 2 — Historical Stock Prices

> *"What were the stock prices for Amazon in Q4 last year?"*

Expected: Agent calls `retrieve_historical_stock_price` with Q4 2025 date range.

In [5]:
response_2 = invoke_agent(
    "What were the stock prices for Amazon in Q4 last year?",
    id_token=id_token,
)

I'll help you get

 Amazon's stock prices for Q

4 of last year (

2023). Let me retrieve the

 historical stock price data for that

...

 performance in Q4 2

023, recovering from October

 volatility and maintaining an

 upward trajectory

 through November and December.

## Cell 6: Query 3 — Cross-Reference Stock + Reports

> *"Compare Amazon's recent stock performance to what analysts predicted in their reports"*

Expected: Agent calls both yfinance (historical) and RAG (earnings reports).

In [6]:
response_3 = invoke_agent(
    "Compare Amazon's recent stock performance to what analysts predicted in their reports",
    id_token=id_token,
)

I'll help you compare

 Amazon's recent stock performance to analyst

 predictions. Let me gather

 the relevant information by looking at

 Amazon's recent financial reports and current

...

 largely

 validate

d by

 actual

 performance.

## Cell 7: Query 4 — Multi-Source Research

> *"I'm researching AMZN -- give me the current price and any relevant information about their AI business"*

Expected: Agent calls realtime tool + RAG tool, combines price with AI business insights.

In [7]:
response_4 = invoke_agent(
    "I'm researching AMZN -- give me the current price"
    " and any relevant information about their AI business",
    id_token=id_token,
)

I'll help you research

 Amazon (AMZ

N) by getting

 the current stock price and finding

 information about their AI business.

...

 reflects investor confidence in Amazon's comprehensive

 AI strategy and the substantial revenue

 growth already being

 generated from their AI investments

.

## Cell 8: Query 5 — Document-Only Query

> *"What is the total amount of office space Amazon owned in North America in 2024?"*

Expected: Agent calls RAG tool only, retrieves data from Amazon 2024 Annual Report.

In [8]:
response_5 = invoke_agent(
    "What is the total amount of office space Amazon owned in North America in 2024?",
    id_token=id_token,
)

I'll search

 Amazon

's financial

 documents to

 find information about their

...

,551 thousand square feet of office

 space in

 the

 region

.

## Cell 9: Multi-Turn Demo

Two messages in the same `thread_id` to demonstrate conversation memory
via `AgentCoreMemorySaver`.

In [9]:
thread_id = str(uuid.uuid4())
print(f"Thread ID: {thread_id}")
print("=" * 60)

print("\n--- Turn 1 ---")
print("User: What is the current stock price of Amazon?\n")
turn_1 = invoke_agent(
    "What is the current stock price of Amazon?",
    id_token=id_token,
    thread_id=thread_id,
)

print("\n--- Turn 2 (follow-up in same thread) ---")
print("User: How does that compare to its price 6 months ago?\n")
turn_2 = invoke_agent(
    "How does that compare to its price 6 months ago?",
    id_token=id_token,
    thread_id=thread_id,
)

Thread ID: d777cd56-66e7-40ae-adc4-51d6597405e1

--- Turn 1 ---
User: What is the current stock price of Amazon?



I'll get

 the current stock price for Amazon (

AMZN) for you.

Amazon

...

 mid-October 

2025 before recovering an

d continuing its gradual upward trajectory

 to today's price

.

## Cell 10: Langfuse Traces

Langfuse captures full traces of every agent invocation — LLM calls, tool
invocations, and graph transitions.

### Langfuse UI Screenshots

**Trace 1: Real-time stock price query** — Full LangGraph span tree with `retrieve_realtime_stock_price` tool call, ChatBedrock LLM invocations, latency (5.89s), cost ($0.008), and graph visualization.

![Langfuse trace — real-time stock price](images/langfuse-trace-realtime.png)

**Trace 2: Multi-turn conversation** — Follow-up question with multiple agent iterations, two `retrieve_historical_stock_price` tool calls, and richer token usage (17.91s, $0.021).

![Langfuse trace — multi-turn conversation](images/langfuse-trace-multiturn.png)

### API Traces

In [10]:
import os

LANGFUSE_PUBLIC_KEY = os.environ.get("LANGFUSE_PUBLIC_KEY", "")
LANGFUSE_SECRET_KEY = os.environ.get("LANGFUSE_SECRET_KEY", "")
LANGFUSE_HOST = os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com")

if LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY:
    from langfuse import Langfuse

    lf = Langfuse(
        public_key=LANGFUSE_PUBLIC_KEY,
        secret_key=LANGFUSE_SECRET_KEY,
        host=LANGFUSE_HOST,
    )

    traces = lf.api.trace.list(page=1, limit=5)
    print(f"Recent Langfuse traces ({len(traces.data)} found):")
    print("-" * 80)
    for t in traces.data:
        print(f"  ID: {t.id}")
        print(f"  Name: {t.name}")
        print(f"  Timestamp: {t.timestamp}")
        print(f"  URL: {LANGFUSE_HOST}/trace/{t.id}")
        print("-" * 80)
else:
    print("Langfuse keys not set — skipping trace retrieval.")
    print("Set LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY to fetch traces.")
    print("\nSee screenshots in notebooks/images/ for trace examples.")

Recent Langfuse traces (5 found):
--------------------------------------------------------------------------------
  ID: 048d4199b8189cca88f52300a8a88ee7
  Name: LangGraph
  Timestamp: 2026-04-09 18:59:45.500000+00:00
  URL: https://cloud.langfuse.com/trace/048d4199b8189cca88f52300a8a88ee7
--------------------------------------------------------------------------------
  ID: e17a347a98e318ac94f479b2b64774b4
  Name: LangGraph
  Timestamp: 2026-04-09 18:59:33.405000+00:00
  URL: https://cloud.langfuse.com/trace/e17a347a98e318ac94f479b2b64774b4
--------------------------------------------------------------------------------
  ID: c87ff7584bc2ab23069c36b579d1f80b
  Name: LangGraph
  Timestamp: 2026-04-09 18:59:06.468000+00:00
  URL: https://cloud.langfuse.com/trace/c87ff7584bc2ab23069c36b579d1f80b
--------------------------------------------------------------------------------
  ID: c185d0e6cfe775579e56c7b8d462d051
  Name: LangGraph
  Timestamp: 2026-04-09 18:58:56.604000+00:00
  URL: http